# Activity 7 Concepts: max_tokens and temperature

Activity 7 asks your Streamlit app to call an LLM and hands you two parameters,
`max_tokens` and `temperature`, that quietly control the quality of the analyst note it
produces. Getting back a reply that stops mid-sentence, or a note that reads differently every
time someone reruns the page, and not knowing which parameter caused it, is a rough way to meet
both for the first time.

This notebook makes both effects visible in isolation, using a small, fixed number of cheap
calls, before you meet them baked into the app. It follows the same setup as Week 6:
`load_dotenv()`, then `OpenAI(api_key=os.environ["OPENAI_API_KEY"])`, model `gpt-4o-mini`.

Load the API key from the repo-root `.env`. Expect `True`, confirming the key loaded,
without ever printing the key text itself.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENAI_API_KEY loaded:", "OPENAI_API_KEY" in os.environ)

This notebook makes real API calls below. Add a small guard so the notebook can still be opened and read start to finish even without a key on hand: if no key was found, the live call cells print a short skip message and move on instead of raising.

In [ ]:
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
if not HAS_KEY:
    print("No OPENAI_API_KEY found. The live call cells below will be skipped. Add your key from Activity 0 to run them for real.")

Create the client. Expect no output, this cell just constructs an object.

In [ ]:
from openai import OpenAI

if HAS_KEY:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## max_tokens: it caps the reply, not the question

A token is roughly three-quarters of an English word: short common words are often one token,
longer or unusual words can be several. `max_tokens` puts a ceiling on the RESPONSE only, the
model still reads your whole question no matter how low it is set.

Ask a real question, but deliberately cripple the reply with `max_tokens=8`. Look for: the
printed reply stopping abruptly, with no closing punctuation, and `finish_reason` reading
`"length"` rather than `"stop"`. `"length"` is the API telling you it did not choose to stop,
it ran out of budget.

In [ ]:
PROMPT = "In three sentences, explain why a single day's taxi ride count might drop far below a typical day."

if HAS_KEY:
    tiny = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=8,
    )
    print(tiny.choices[0].message.content)
    print("finish_reason:", tiny.choices[0].finish_reason)
else:
    print("Skipped (no API key): this cell would call the API with max_tokens=8.")

Notice what did NOT happen above: the model did not shorten or summarize its answer to
fit inside 8 tokens. It started writing a full answer and got cut off wherever the budget ran
out, mid-word or mid-clause. A too-small `max_tokens` truncates, it never compresses.

Ask the exact same question again, this time with a realistic `max_tokens=120`. Look
for: a complete answer ending in punctuation, `finish_reason` back to `"stop"`, and
`response.usage` showing real token counts, `prompt_tokens` for your question and
`completion_tokens` for the reply.

In [ ]:
if HAS_KEY:
    reasonable = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=120,
    )
    print(reasonable.choices[0].message.content)
    print("finish_reason:", reasonable.choices[0].finish_reason)
    print(reasonable.usage)
else:
    print("Skipped (no API key): this cell would call the API with max_tokens=120.")

The exact numbers above will vary a little run to run, since the model rarely writes the
identical wording twice. The shape is the point, not the exact count: a short, three-sentence
factual answer costs on the order of dozens to a couple hundred completion tokens, so
`max_tokens=120` was already comfortably enough for this prompt.

## temperature: how much the model is allowed to vary

`temperature` controls how randomly the model samples its next word at each step.
`temperature=0` makes it pick close to the single most likely next word every time, so the same
prompt tends to produce close to the same answer on repeat calls.

Call the same short prompt twice at `temperature=0`. Look for: the two replies to read the same
or very nearly the same.

In [ ]:
SHORT_PROMPT = "In five words or fewer, name one plausible reason for a one-day drop in city taxi ridership."

if HAS_KEY:
    cold_1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": SHORT_PROMPT}],
        max_tokens=20,
        temperature=0,
    )
    cold_2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": SHORT_PROMPT}],
        max_tokens=20,
        temperature=0,
    )
    print("Run 1:", cold_1.choices[0].message.content)
    print("Run 2:", cold_2.choices[0].message.content)
else:
    print("Skipped (no API key): this cell would call the API twice at temperature=0.")

What to look for above: Run 1 and Run 2 should match closely. If they are not
byte-for-byte identical, that is still normal, `temperature=0` makes sampling greedy, not a
lifetime guarantee of an identical reply (providers batch requests behind the scenes in ways
you cannot see), but the two runs should land on the same idea in the same words or close to
it.

Now call the exact same prompt twice more at `temperature=1.5` (this API's scale runs from 0 to
2). Look for: the two replies to genuinely diverge this time, different reasons, different
wording, possibly a different length.

In [ ]:
if HAS_KEY:
    hot_1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": SHORT_PROMPT}],
        max_tokens=20,
        temperature=1.5,
    )
    hot_2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": SHORT_PROMPT}],
        max_tokens=20,
        temperature=1.5,
    )
    print("Run 3:", hot_1.choices[0].message.content)
    print("Run 4:", hot_2.choices[0].message.content)
else:
    print("Skipped (no API key): this cell would call the API twice at temperature=1.5.")

Compare what you saw in the `temperature=0` pair against the `temperature=1.5` pair.
The low-temperature pair should have stayed close together; the high-temperature pair should
have visibly spread apart, in reason, wording, or both.

This is exactly why Activity 7's analyst note uses a low temperature: a factual summary of
computed anomalies should read the same way every time a manager refreshes the page, not turn
into a different story on every rerun.

## Low temperature buys consistency, not correctness

Here is the trap: `temperature=0` made the model answer the same way twice, but "the same
wrong answer, twice" is still wrong. A low temperature only makes the model more likely to
repeat its single most probable answer, it does not check whether that answer is true. If the
top answer happens to be a fabricated number or an invented cause, `temperature=0` will
cheerfully repeat that fabrication every time you ask.

This is exactly why Activity 7 does not stop at "ask the model to write a note." It hands the
model ONLY a small, pre-computed facts block, never the raw data, and then makes fact-checking
the draft against that facts block a mandatory step before anyone would hand the note to a
manager.

## Cost: what six small calls actually cost

Every `response.usage` object carries `prompt_tokens`, `completion_tokens`, and
`total_tokens`. Add up the totals across every call made in this notebook.

In [ ]:
if HAS_KEY:
    calls = [tiny, reasonable, cold_1, cold_2, hot_1, hot_2]
    total_tokens = sum(r.usage.total_tokens for r in calls)
    print(f"{len(calls)} calls, {total_tokens} tokens total across this notebook")
else:
    print("Skipped (no API key): nothing to total.")

`gpt-4o-mini` is priced per million tokens, with input and output priced separately, and
both rates are a small fraction of a cent per call at the sizes used here. Six short calls,
each capped between 8 and 120 completion tokens, costs well under a cent to run end to end,
including every call you just made above. Check OpenAI's current pricing page before assuming
a specific rate, prices change over time.

**Try changing this and re-run, no new code needed:**

1. In the `max_tokens=8` cell, change it to `max_tokens=3` and re-run. Does the reply still
   form a complete word, or does it cut off mid-word this time too?
2. In the two `temperature=1.5` calls, change `1.5` to `2.0` (the top of this API's range) and
   re-run both. Do the two replies diverge even further apart, or does one of them turn into
   text that barely reads as an answer at all?